# Bengali Hallucination Detection - Gemma 12B QAT GGUF Submission

This notebook is the local best run for the datathon workspace. It uses the LM Studio Gemma 12B QAT GGUF copied into this repo, served through LM Studio's bundled `llama.cpp` server, then applies a validated context-grounding post-process.

Validation on the 299 labeled sample rows:

- Raw Gemma 12B QAT: macro-F1 `0.7440`, hallucinated-class F1 `0.7226`
- Gemma + context exact-support rule: macro-F1 `0.7590`, hallucinated-class F1 `0.7300`

Final local output: `outputs/submission_gemma12_qat_context_rule.csv` and root `submission.csv`.


In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import time
import urllib.request

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

ROOT = Path.cwd()
DATA_DIR = ROOT / "bengali-hallucination"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_PATH = ROOT / "models/lmstudio/gemma-4-12B-it-QAT-GGUF/gemma-4-12B-it-QAT-Q4_0.gguf"
LLAMA_SERVER = Path("/Users/seyam/.lmstudio/extensions/backends/llama.cpp-mac-arm64-apple-metal-advsimd-2.23.1/llama-server")
HOST = "127.0.0.1"
PORT = 8099
SERVER_URL = f"http://{HOST}:{PORT}/v1/chat/completions"

SAMPLE_PATH = DATA_DIR / "dataset samples.json"
TEST_PATH = DATA_DIR / "test set.csv"

assert MODEL_PATH.exists(), MODEL_PATH
assert LLAMA_SERVER.exists(), LLAMA_SERVER
print("Model:", MODEL_PATH)
print("Server:", LLAMA_SERVER)


In [ ]:
NO_CONTEXT = {"", "[NULL]", "NULL", "null", "nan", "NaN", "None"}
BN_DIGITS = str.maketrans("০১২৩৪۵۶۷۸۹۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")


def clean_context(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    return "" if text in NO_CONTEXT else text


def clean_text(value):
    return "" if pd.isna(value) else str(value).strip()


def normalize_bn(text):
    text = clean_text(text).translate(BN_DIGITS).lower()
    text = re.sub(r"[‌‍]", "", text)
    text = re.sub(r'[।,;:!?()\[\]{}"“”‘’`*_\-–—/\\]+', ' ', text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def response_in_context(context, response):
    response_norm = normalize_bn(response)
    return bool(response_norm and response_norm in normalize_bn(context))


def parse_label(text):
    text = (text or "").strip().translate(BN_DIGITS)
    match = re.search(r"(?<!\d)([01])(?!\d)", text)
    return int(match.group(1)) if match else 0


def prepare_frame(df):
    df = df.copy()
    for col in ["prompt_bn", "response_bn"]:
        df[col] = df[col].map(clean_text)
    df["context"] = df["context"].map(clean_context)
    df["has_context"] = df["context"].str.len() > 0
    return df


In [ ]:
FEWSHOT = """তুমি বাংলা hallucination detector। শুধু 0 অথবা 1 লিখবে।
0 = উত্তরটি ভুল, বানানো, প্রশ্নের সঙ্গে মেলে না, অথবা প্রসঙ্গ দ্বারা সমর্থিত নয়।
1 = উত্তরটি সঠিক এবং প্রসঙ্গ থাকলে প্রসঙ্গ দ্বারা সমর্থিত।

প্রশ্ন: বাংলাদেশের রাজধানী কী?
উত্তর: ঢাকা
লেবেল: 1

প্রশ্ন: বাংলাদেশের রাজধানী কী?
উত্তর: চট্টগ্রাম
লেবেল: 0

প্রশ্ন: "ধান্ধা" এর ভাবার্থ কী?
উত্তর: কোন অসৎ উদ্দেশ্য
লেবেল: 1

প্রশ্ন: শেখ মুজিবুর রহমান কোথায় জন্মগ্রহণ করেন?
উত্তর: চুরুলিয়া, বর্ধমান
লেবেল: 0

প্রসঙ্গ: অভ্র কীবোর্ড সফটওয়্যারটি মেহদী হাসান খান তৈরি করেন।
প্রশ্ন: অভ্র কিবোর্ড কে উদ্ভাবন করেন?
উত্তর: মেহদী হাসান খান
লেবেল: 1

প্রসঙ্গ: প্রথম চলচ্চিত্র সোনার বেড়ি এবং সর্বশেষ পূর্ণদৈর্ঘ্য চলচ্চিত্র রানওয়ে।
প্রশ্ন: সর্বশেষ চলচ্চিত্রটি কত সালে মুক্তি পায়?
উত্তর: রানওয়ে
লেবেল: 0
"""


def make_prompt(row, max_context_chars=2200):
    parts = [
        FEWSHOT,
        "এখন নতুন উদাহরণের লেবেল দাও। ব্যাখ্যা নয়, শুধু 0 অথবা 1।",
    ]
    if row.context:
        parts.append("প্রসঙ্গ: " + row.context[:max_context_chars])
    parts.append("প্রশ্ন: " + row.prompt_bn)
    parts.append("উত্তর: " + row.response_bn)
    parts.append("লেবেল:")
    return "\n".join(parts)


In [ ]:
def wait_for_server(timeout=180):
    start = time.time()
    while time.time() - start < timeout:
        try:
            payload = {
                "model": "local",
                "messages": [{"role": "user", "content": "শুধু 1 লিখো।"}],
                "temperature": 0,
                "max_tokens": 4,
                "stream": False,
                "chat_template_kwargs": {"enable_thinking": False},
            }
            request = urllib.request.Request(
                SERVER_URL,
                data=json.dumps(payload).encode("utf-8"),
                headers={"Content-Type": "application/json"},
            )
            with urllib.request.urlopen(request, timeout=10) as response:
                if response.status == 200:
                    return True
        except Exception:
            time.sleep(2)
    return False


def start_llama_server():
    if wait_for_server(timeout=5):
        print("Using already-running llama-server")
        return None

    cmd = [
        str(LLAMA_SERVER),
        "-m", str(MODEL_PATH),
        "--host", HOST,
        "--port", str(PORT),
        "-c", "2048",
        "-ngl", "auto",
        "--log-disable",
    ]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if not wait_for_server(timeout=240):
        process.terminate()
        raise RuntimeError("llama-server did not become ready")
    print("Started llama-server on", SERVER_URL)
    return process

server_process = start_llama_server()


In [ ]:
def call_gemma(prompt, retries=3):
    payload = {
        "model": "local",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
        "max_tokens": 8,
        "stream": False,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    data = json.dumps(payload).encode("utf-8")
    for attempt in range(retries):
        try:
            request = urllib.request.Request(
                SERVER_URL,
                data=data,
                headers={"Content-Type": "application/json"},
            )
            with urllib.request.urlopen(request, timeout=240) as response:
                output = json.loads(response.read())
            message = output["choices"][0]["message"]
            text = message.get("content") or message.get("reasoning_content") or ""
            return text, output.get("usage", {})
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(3)


def run_gemma(df, trace_path, partial_submission_path=None, id_values=None):
    trace_path = Path(trace_path)
    existing = []
    if trace_path.exists():
        existing = [json.loads(line) for line in trace_path.read_text(encoding="utf-8").splitlines() if line.strip()]

    preds = [int(row["pred"]) for row in existing]
    start_idx = len(existing)
    print(f"Resume from {start_idx}/{len(df)}")

    start = time.time()
    with trace_path.open("a", encoding="utf-8") as handle:
        for i, row in enumerate(df.itertuples()):
            if i < start_idx:
                continue
            generated, usage = call_gemma(make_prompt(row))
            pred = parse_label(generated)
            record = {"i": i, "pred": int(pred), "gen": generated, "usage": usage}
            if id_values is not None:
                record["id"] = int(id_values[i])
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            handle.flush()
            preds.append(pred)

            if (i + 1) % 50 == 0:
                elapsed = (time.time() - start) / 60
                print(f"done {i + 1}/{len(df)} elapsed_min={elapsed:.1f} counts={pd.Series(preds).value_counts().to_dict()}")
                if partial_submission_path is not None and id_values is not None:
                    pd.DataFrame({"id": id_values[:len(preds)], "label": preds}).to_csv(partial_submission_path, index=False)

    return np.array(preds, dtype=int)


In [ ]:
with SAMPLE_PATH.open(encoding="utf-8") as handle:
    sample_df = pd.DataFrame(json.load(handle))
sample_df = prepare_frame(sample_df)

sample_trace = OUTPUT_DIR / "gemma12_qat_sample_trace.jsonl"
sample_pred = run_gemma(sample_df, sample_trace)
y = sample_df["label"].to_numpy()

print("Raw Gemma")
print("accuracy", accuracy_score(y, sample_pred))
print("macro_f1", f1_score(y, sample_pred, average="macro"))
print("f1_label_0", f1_score(y, sample_pred, pos_label=0))
print("f1_label_1", f1_score(y, sample_pred, pos_label=1))
print(confusion_matrix(y, sample_pred))
print(classification_report(y, sample_pred))


In [ ]:
def apply_context_rule(df, labels):
    labels = np.asarray(labels, dtype=int).copy()
    has_context = df["has_context"].to_numpy()
    exact_support = np.array([
        response_in_context(row.context, row.response_bn)
        for row in df.itertuples()
    ])
    labels[has_context & exact_support] = 1
    labels[has_context & ~exact_support] = 0
    return labels

sample_post = apply_context_rule(sample_df, sample_pred)
print("Gemma + context exact-support rule")
print("accuracy", accuracy_score(y, sample_post))
print("macro_f1", f1_score(y, sample_post, average="macro"))
print("f1_label_0", f1_score(y, sample_post, pos_label=0))
print("f1_label_1", f1_score(y, sample_post, pos_label=1))
print(confusion_matrix(y, sample_post))
print(classification_report(y, sample_post))
print("changed sample rows", int((sample_post != sample_pred).sum()))


In [ ]:
test_df = prepare_frame(pd.read_csv(TEST_PATH))

test_trace = OUTPUT_DIR / "gemma12_qat_test_trace.jsonl"
partial_path = OUTPUT_DIR / "submission_gemma12_qat_partial.csv"
test_pred = run_gemma(test_df, test_trace, partial_submission_path=partial_path, id_values=test_df["id"].to_numpy())

raw_submission = pd.DataFrame({"id": test_df["id"].to_numpy(), "label": test_pred})
raw_submission.to_csv(OUTPUT_DIR / "submission_gemma12_qat.csv", index=False)
print(raw_submission["label"].value_counts().to_dict())
raw_submission.head()


In [ ]:
final_pred = apply_context_rule(test_df, test_pred)
submission = pd.DataFrame({"id": test_df["id"].to_numpy(), "label": final_pred})
submission.to_csv(OUTPUT_DIR / "submission_gemma12_qat_context_rule.csv", index=False)
submission.to_csv(ROOT / "submission.csv", index=False)

print("Final counts", submission["label"].value_counts().to_dict())
print("Changed vs raw Gemma", int((submission["label"].to_numpy() != test_pred).sum()))
print("Wrote", OUTPUT_DIR / "submission_gemma12_qat_context_rule.csv")
print("Wrote", ROOT / "submission.csv")
submission.head()


In [ ]:
# Optional cleanup if this notebook started the local server.
if server_process is not None:
    server_process.terminate()
    try:
        server_process.wait(timeout=10)
        print("Stopped llama-server")
    except subprocess.TimeoutExpired:
        server_process.kill()
        print("Killed llama-server")
